# Read and plot the original source

Same complete `moi10_all` readout and plotting options as `read_atst.ipynb`. Source ranges and condition annotations remain explicit; no ATST file is read. Matplotlib plots the source values directly.


In [ ]:
from pathlib import Path
import math
import matplotlib.pyplot as plt
import pandas as pd
from openpyxl import load_workbook

readout_id = 'moi10_all'
rows, curves = [], []

with pd.ExcelFile("read_shared_data/sources/Plate reader/Plate_reader_raw_MOI10.xlsx") as book:
    for sheet in book.sheet_names:
        raw = pd.read_excel(book, sheet_name=sheet, header=None, dtype=str, keep_default_na=False)
        raw = raw.loc[raw.ne("").any(axis=1)]
        # Reuse curated source-column annotations, not the exported ATST layout.
        layout = pd.read_csv(Path("read_shared_data/layout_inputs") / ("layout_moi10_" + sheet.replace(" ", "_").replace("-", "_") + ".tsv"), sep="\t", dtype=str, keep_default_na=False)
        time = pd.to_numeric(raw.iloc[1:, 0]).to_numpy()
        for column, row in enumerate(layout.to_dict("records"), 1):
            rows.append(row)
            curves.append(pd.Series(pd.to_numeric(raw.iloc[1:, column]).to_numpy(), index=time))

# Local curve IDs preserve source order; different time grids align without interpolation.
layout = pd.DataFrame(rows)
layout["curve_id"] = [f"curve_{i}" for i in range(1, len(rows) + 1)]
readings = pd.concat(curves, axis=1).sort_index()
readings.columns = layout["curve_id"]
readings = readings.rename_axis("Time").reset_index()


In [ ]:
panels = list(layout.groupby('isolate_id', sort=False))
ncols = min(2, len(panels))
nrows = math.ceil(len(panels) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows), squeeze=False)
palette = plt.rcParams["axes.prop_cycle"].by_key()["color"]
condition_colors = {}

for ax, (panel, panel_layout) in zip(axes.flat, panels):
    for condition, group in panel_layout.groupby(['phage_id', 'moi'], sort=False):
        if condition not in condition_colors:
            condition_colors[condition] = palette[len(condition_colors) % len(palette)]
        color = condition_colors[condition]
        label = " | ".join(str(value) for value in condition)
        values = readings[group["curve_id"]]
        for index, curve_id in enumerate(group["curve_id"]):
            ax.plot(
                readings["Time"], readings[curve_id], color=color, alpha=0.6,
                label=label if index == 0 else None,
            )
    ax.set_xlabel("Time (h)")
    ax.set_ylabel("od600")
    panel_label = "<blank>" if panel == "" else str(panel)
    ax.set_title(f"{readout_id} | isolate_id={panel_label}")
    ax.legend(frameon=False)

for ax in list(axes.flat)[len(panels):]:
    fig.delaxes(ax)
fig.tight_layout()
plt.show()


Individual source curves are shown; the merged readout repeats control replicate IDs within some isolate/condition groups.
